In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

from sklearn.model_selection import  train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import  StandardScaler, MinMaxScaler

import torch
from torch import nn
from torch.utils.data import  DataLoader, TensorDataset
from torch.optim import Adam

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams['font.family'] = 'NanumGothic' # Windows
# plt.rcParams['font.family'] = 'AppleGothic' # Mac
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 1. 데이터 준비 (정답: w=3, b=2)
X = np.array([1, 2, 3, 4, 5])
y = np.array([5, 8, 11, 14, 17])

# 가중치와 편향 초기화 (아직은 아무것도 모르는 상태)
w = 0.0
b = 0.0
learning_rate = 0.02
epochs = 1500

print("학습 시작...")

for epoch in range(epochs):
    # --- [Step 1: 순전파 (Forward)] ---
    # 현재의 w, b로 예측값 계산
    y_pred = w * X + b
    
    # --- [Step 2: 오차 계산 (Loss)] ---
    # MSE(평균제곱오차) 계산
    loss = np.mean((y_pred - y) ** 2)
    
    # --- [Step 3: 역전파 (Backpropagation - 기울기 계산)] ---
    # 손실 함수를 w와 b로 편미분하여 기울기(Gradient) 도출
    w_gradient = 2 * np.mean((y_pred - y) * X)
    b_gradient = 2 * np.mean(y_pred - y)
    
    # --- [Step 4: 경사하강법 (Gradient Descent - 업데이트)] ---
    # 기울기의 반대 방향으로 가중치 수정
    w = w - (learning_rate * w_gradient)
    b = b - (learning_rate * b_gradient)
    
    # 100번마다 학습 경과 출력
    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss = {loss:.4f}, w = {w:.4f}, b = {b:.4f}")

print("-" * 30)
print(f"최종 결과: w = {w:.2f}, b = {b:.2f}")
print(f"예측 확인 (x=10): {w * 10 + b:.2f} (정답: 32)")


학습 시작...
Epoch 0: Loss = 139.0000, w = 1.5600, b = 0.4400
Epoch 100: Loss = 0.0552, w = 3.1516, b = 1.4528
Epoch 200: Loss = 0.0142, w = 3.0769, b = 1.7224
Epoch 300: Loss = 0.0037, w = 3.0390, b = 1.8591
Epoch 400: Loss = 0.0009, w = 3.0198, b = 1.9285
Epoch 500: Loss = 0.0002, w = 3.0100, b = 1.9637
Epoch 600: Loss = 0.0001, w = 3.0051, b = 1.9816
Epoch 700: Loss = 0.0000, w = 3.0026, b = 1.9907
Epoch 800: Loss = 0.0000, w = 3.0013, b = 1.9953
Epoch 900: Loss = 0.0000, w = 3.0007, b = 1.9976
Epoch 1000: Loss = 0.0000, w = 3.0003, b = 1.9988
Epoch 1100: Loss = 0.0000, w = 3.0002, b = 1.9994
Epoch 1200: Loss = 0.0000, w = 3.0001, b = 1.9997
Epoch 1300: Loss = 0.0000, w = 3.0000, b = 1.9998
Epoch 1400: Loss = 0.0000, w = 3.0000, b = 1.9999
------------------------------
최종 결과: w = 3.00, b = 2.00
예측 확인 (x=10): 32.00 (정답: 32)


In [30]:
def make_DataSet(X_train, X_val, y_train, y_val, batch_size = 32) :
    # batch_size : 모델이 한 번에 몇 개의 데이터를 보고 학습할지 정하는 값

    # 데이터 텐서로 변환
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

    # TensorDataset 생성 : 텐서 데이터셋으로 합치기
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

    # DataLoader 생성
    train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = False)

    return train_loader, X_val_tensor, y_val_tensor

def train(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    tr_loss = 0
    model.train()                                           # 훈련 모드로 설정
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)


        pred = model(X)
        loss = loss_fn(pred, y)
        tr_loss += loss


        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    tr_loss /= num_batches

    return tr_loss.item()

def evaluate(X_val_tensor, y_val_tensor, model, loss_fn, device):
    model.eval()

    with torch.no_grad():
        X, y = X_val_tensor.to(device), y_val_tensor.to(device)
        pred = model(X)
        eval_loss = loss_fn(pred, y).item()

    return eval_loss

device = 'cpu'
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # 멀티 GPU 사용 시
    torch.backends.cudnn.deterministic = True # 연산 결과 고정
    torch.backends.cudnn.benchmark = False # 최적화 알고리즘 고정
# 행운의 숫자 하나를 골라 시드를 고정합니다.
set_seed(42)

In [35]:
advertising = pd.read_csv('content/advertising.csv')

target = 'Sales'
X = advertising.drop(target, axis=1)
y = advertising[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_loader, X_test_ts, y_test_ts = make_DataSet(X_train, X_test, y_train, y_test, 4)

n_feature = X.shape[1]

# 모델 구조 설계
model = nn.Sequential(
            nn.Linear(n_feature, 3),
            nn.ReLU(),
            nn.Linear(3,1)
            ).to(device)

loss_fn = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=0.001)   # model.parameters() : 모델의 가중치와 편향

epochs = 150
for t in range(1, epochs + 1):
    tr_loss = train(train_loader, model, loss_fn, optimizer, device)
    test_loss = evaluate(X_test_ts, y_test_ts, model, loss_fn, device)

    if t % 10 == 0:
        print(f"Epoch {t}: Loss = {tr_loss:.4f}, test_loss = {test_loss:.4f}")

evaluate(X_test_ts, y_test_ts, model, loss_fn, device)

Epoch 10: Loss = 147.2314, test_loss = 156.1927
Epoch 20: Loss = 65.8619, test_loss = 67.4240
Epoch 30: Loss = 19.9752, test_loss = 18.4292
Epoch 40: Loss = 9.7221, test_loss = 7.2697
Epoch 50: Loss = 8.5180, test_loss = 5.8629
Epoch 60: Loss = 7.9608, test_loss = 5.3887
Epoch 70: Loss = 7.3892, test_loss = 4.9696
Epoch 80: Loss = 6.8042, test_loss = 4.5548
Epoch 90: Loss = 6.2259, test_loss = 4.1537
Epoch 100: Loss = 5.6735, test_loss = 3.7787
Epoch 110: Loss = 5.1636, test_loss = 3.4398
Epoch 120: Loss = 4.7087, test_loss = 3.1438
Epoch 130: Loss = 4.3167, test_loss = 2.8937
Epoch 140: Loss = 3.9904, test_loss = 2.6889
Epoch 150: Loss = 3.7281, test_loss = 2.5264


2.526444911956787

In [ ]:
df = pd.read_csv('content/boston.csv')
df.head()

target = 'medv'
X = df.drop(target, axis=1)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_loader, X_test_ts, y_test_ts = make_DataSet(X_train, X_test, y_train, y_test, 16)

n_feature = X.shape[1]

# 모델 구조 설계
model = nn.Sequential(
            nn.Linear(n_feature, 64),
            nn.ReLU(),
            nn.Dropout(0.2),  # 20%의 뉴런을 랜덤하게 끔 (암기 방지)
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),  # 여기도 추가
            nn.Linear(32, 1)
            ).to(device)

loss_fn = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

epochs = 300
for t in range(1, epochs + 1):
    tr_loss = train(train_loader, model, loss_fn, optimizer, device)
    test_loss = evaluate(X_test_ts, y_test_ts, model, loss_fn, device)

    if t % 30 == 0:
        print(f"Epoch {t}: Loss = {tr_loss:.4f}, test_loss = {test_loss:.4f}")

evaluate(X_test_ts, y_test_ts, model, loss_fn, device)

Epoch 30: Loss = 33.3903, test_loss = 18.7569
Epoch 60: Loss = 28.4536, test_loss = 13.2207
Epoch 90: Loss = 29.1203, test_loss = 11.1692
Epoch 120: Loss = 27.4936, test_loss = 11.1113
Epoch 150: Loss = 26.6142, test_loss = 10.6122
Epoch 180: Loss = 25.1591, test_loss = 9.6645
Epoch 210: Loss = 21.8718, test_loss = 9.9687
Epoch 240: Loss = 24.9515, test_loss = 8.9162
Epoch 270: Loss = 20.8967, test_loss = 9.1203
Epoch 300: Loss = 20.8850, test_loss = 9.7607


9.760712623596191